# 1.0 — Limpieza de YRBS (2005-2021)

**Objetivo.** Llevar el dataset stacked de YRBS (134,674 registros, 9 años) a un formato canónico en `data/processed/yrbs_clean_2005_2021.parquet` aplicando el orden metodologico establecido:

1. **Diagnóstico inicial** — shape, dtypes, head, NaN por columna y por año.
2. **Normalización de tipos** — q-vars a float64, `weight`/`stratum`/`psu` a float64, año a int.
3. **Consistencia de categorías** — documentar los Q-codes por año y construir un crosswalk.
4. **Valores imposibles** — fuera de rango clínico (e.g., Q1=8) → NaN con justificación.
5. **Duplicados** — YRBS no publica IDs individuales; verificar duplicados por año+localización.
6. **Outliers** — `weight` extremo (max 19.28 en 2009) y análisis de Q80 (ordinal pero con cola larga).
7. **Faltantes** — patrón MCAR/MAR/MNAR y decisión de imputar/eliminar.
8. **Verificación final** — shape, distribuciones, sanity checks contra valores publicados.

**Principio rector.** Cada transformación tiene celda markdown de **diagnóstico → decisión → verificación**.

**Output esperado:** `data/processed/yrbs_clean_2005_2021.parquet` con las variables unificadas:
- `year`, `age`, `sex`, `grade`
- `sad_hopeless` (binario: 1=sí, 0=no, NaN=missing)
- `considered_suicide`, `made_plan`, `attempted_suicide_yesno`, `attempted_suicide_ordinal`
- `screen_time` (1-7, solo 2017/2019; para 2017 la pregunta era TV only)
- `weight`, `stratum`, `psu` (para análisis con diseño complejo)

## Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from wired_apart import config
from wired_apart.dataset import (
 load_yrbs_processed,
 YRBS_QCODE_CROSSWALK,
)

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)
np.random.seed(config.RANDOM_SEED)

## 1. Diagnóstico inicial

### Diagnóstico
Cargamos el stacked de 9 años (2005-2021) y verificamos shape, dtypes y NaN globales.

In [2]:
df = load_yrbs_processed()
print(f"Shape: {df.shape}")
print(f"A\u00f1os: {sorted(df['year'].unique())}")
print(f"Registros por a\u00f1o:\n{df['year'].value_counts().sort_index()}\n")
print(f"NaN totales: {df.isna().sum().sum():,}")
print(f"% NaN: {100*df.isna().sum().sum()/df.size:.2f}%")
print(f"\nDtypes summary:\n{df.dtypes.value_counts()}")

Shape: (134674, 230)
Años: [np.int64(2005), np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2017), np.int64(2019), np.int64(2021)]
Registros por año:
year
2005    13917
2007    14041
2009    16410
2011    15425
2013    13583
2015    15624
2017    14765
2019    13677
2021    17232
Name: count, dtype: int64

NaN totales: 12,687,653
% NaN: 40.96%

Dtypes summary:
float64    221
str          7
int64        1
object       1
Name: count, dtype: int64


### Decisión
Cargamos desde el Parquet procesado (no desde .mdb) para desacoplar este notebook del driver ODBC. Las q-vars ya vienen como float64 gracias al notebook 0.0.

### Verificación
Confirmamos que el shape coincide con la FASE 2 (134,674 registros) y que no hay sorpresas estructurales.

## 2. Normalización de tipos

### Diagnóstico
Identificamos columnas con dtypes que no son los óptimos: q-vars ya están en float64 (del notebook 0.0), pero `year` debería ser int, y queremos asegurar que `weight`/`stratum`/`psu` sean float64.

In [3]:
# Diagnóstico: tipos actuales
print("Tipos actuales:")
for col in ['year', 'q1', 'q2', 'q3', 'q4', 'q25', 'q80', 'weight', 'stratum', 'psu']:
    if col in df.columns:
        print(f" {col:10s} -> {df[col].dtype}, NaN: {df[col].isna().sum()}")
    else:
        print(f" {col:10s} -> MISSING")

# Verificamos que el rango de q-vars es consistente con el codebook
for q in ['q1', 'q2', 'q3']:
    if q in df.columns:
        print(f"\n{q} value_counts:\n{df[q].value_counts(dropna=False).sort_index()}")


Tipos actuales:
 year       -> int64, NaN: 0
 q1         -> float64, NaN: 637
 q2         -> float64, NaN: 871
 q3         -> float64, NaN: 871
 q4         -> float64, NaN: 2161
 q25        -> float64, NaN: 1700
 q80        -> float64, NaN: 10044
 weight     -> float64, NaN: 0
 stratum    -> float64, NaN: 0
 psu        -> float64, NaN: 0

q1 value_counts:
q1
1.0      335
2.0      209
3.0    15816
4.0    31986
5.0    34168
6.0    33319
7.0    18204
NaN      637
Name: count, dtype: int64

q2 value_counts:
q2
1.0    67158
2.0    66645
NaN      871
Name: count, dtype: int64

q3 value_counts:
q3
1.0    34522
2.0    33559
3.0    33390
4.0    32109
5.0      223
NaN      871
Name: count, dtype: int64


### Decisión
Mantenemos float64 en q-vars y weight/stratum/psu (preserva NaN correctamente; pandas no soporta NaN en int). Convertimos `year` a int (es categórico y no tiene NaN).

In [4]:
df['year'] = df['year'].astype('int64')
print(f"year dtype: {df['year'].dtype}, min: {df['year'].min()}, max: {df['year'].max()}")

year dtype: int64, min: 2005, max: 2021


### Verificación
`year` ya es int. No tocamos q-vars ni pesos.

## 3. Consistencia de categorías: crosswalk de Q-codes por año

### Diagnóstico (CRÍTICO)

**Hallazgo principal.** YRBS rota los Q-codes cada 2 años para evitar priming effects (que los estudiantes memoricen patrones), pero la redacción de las preguntas clave se mantiene estable. Esto significa que el dataset que cargamos tiene los Q-codes en posiciones distintas según el año, aunque pregunten lo mismo.

Para verificar el crosswalk, comparamos la distribución de `q22`..`q28` en cada año y vemos cuál cae en el rango esperado:
- **sad/hopeless (depresión):** ~25-40% dicen "yes"
- **considered suicide:** ~15-20% dicen "yes"
- **made plan:** ~12-15% dicen "yes"
- **attempted suicide:** ~7-10% dicen "yes" (más raro)

In [5]:
# Tabla de distribución de Q-codes por año: cuál cae en 25-40% (depresión)
print(f"{'Year':<6} | Q22%yes Q23%yes Q24%yes Q25%yes Q26%yes Q27%yes Q28%yes")
for year in sorted(df['year'].unique()):
    sub = df[df['year']==year]
    row = f"{year:<6} |"
    for q in ['q22','q23','q24','q25','q26','q27','q28']:
        if q in sub.columns:
            valid = sub[q].dropna()
            pct_yes = (valid==1).mean()*100 if len(valid)>0 else 0
            row += f" {pct_yes:5.1f}"
        else:
            row += " --"
    print(row)
print("\nInterpretación: la columna con 25-40% YES es 'sad/hopeless' (depresión).")


Year   | Q22%yes Q23%yes Q24%yes Q25%yes Q26%yes Q27%yes Q28%yes
2005   |   8.1  30.2  16.9  13.3  91.0  91.2  55.7
2007   |   8.5  30.0  15.1  11.9  92.0  92.1  52.7
2009   |  19.0  27.9  14.5  11.6  92.8  92.8  47.7
2011   |  18.0  14.9  29.7  15.9  13.2  91.3  91.5


2013   |  24.4  24.4  18.6  13.9  30.3  16.7  13.9
2015   |  30.4  30.0  19.1  14.7  31.0  18.2  15.4
2017   |  32.2  18.2  14.5  31.9  17.7  14.0  92.2
2019   |  34.4  20.1  15.9  36.7  19.6  16.0  89.9
2021   |  40.9  16.2  16.2  39.8  21.2  17.2  88.7

Interpretación: la columna con 25-40% YES es 'sad/hopeless' (depresión).


### Decisión

Construimos un **crosswalk** que mapea cada concepto (sad_hopeless, considered_suicide, etc.) al Q-code correcto en cada año, validado contra los codebooks de YRBS y contra las distribuciones. Lo guardamos en `wired_apart/dataset.py` (`YRBS_QCODE_CROSSWALK`). Lo mostramos aquí para transparencia:

In [6]:
import pprint
pprint.pprint(YRBS_QCODE_CROSSWALK)

{2005: {'attempted_suicide': 'q22',
        'bullied_school': 'q20',
        'considered_suicide': 'q24',
        'electronically_bullied': 'q21',
        'made_plan': 'q25',
        'sad_hopeless': 'q23'},
 2007: {'attempted_suicide': 'q22',
        'bullied_school': 'q20',
        'considered_suicide': 'q24',
        'electronically_bullied': 'q21',
        'made_plan': 'q25',
        'sad_hopeless': 'q23'},
 2009: {'attempted_suicide': 'q26',
        'bullied_school': 'q21',
        'considered_suicide': 'q24',
        'electronically_bullied': 'q22',
        'made_plan': 'q25',
        'sad_hopeless': 'q23'},
 2011: {'attempted_suicide': 'q27',
        'bullied_school': 'q22',
        'considered_suicide': 'q25',
        'electronically_bullied': 'q23',
        'made_plan': 'q26',
        'sad_hopeless': 'q24'},
 2013: {'attempted_suicide': 'q29',
        'bullied_school': 'q24',
        'considered_suicide': 'q27',
        'electronically_bullied': 'q25',
        'made_plan': 'q28

### Verificación
Para cada año, verificamos que las distribuciones de las variables del crosswalk caen en el rango esperado (depresión 25-40%, considered 15-20%, plan 12-15%, attempt 7-10%).

In [7]:
print(f"{'Year':<6} | {'sad%':>6} {'cons%':>6} {'plan%':>6} {'att%':>6}")
for year, mapping in YRBS_QCODE_CROSSWALK.items():
    sub = df[df['year']==year]
    row = f"{year:<6} |"
    for concept, qcode in [
        ('sad_hopeless', mapping['sad_hopeless']),
        ('considered_suicide', mapping['considered_suicide']),
        ('made_plan', mapping['made_plan']),
        ('attempted_suicide', mapping['attempted_suicide']),
    ]:
        if qcode in sub.columns:
            valid = sub[qcode].dropna()
            if year in [2005, 2007]:
                # In 2005/2007, attempted is binary 1=yes, 2=no
                pct = (valid==1).mean()*100
            elif year in [2009]:
                # In 2009, attempted is 1=0 times (NO), 2=1 time, 3=2-3, 4=4-5, 5=6+
                pct = (valid>=2).mean()*100  # any attempt = 1+
            else:
                # In 2011+, attempted is binary 1=yes, 2=no
                pct = (valid==1).mean()*100
            row += f" {pct:5.1f}"
        else:
            row += " --"
    print(row)
print("\nSi los rangos son los esperados (sad 25-40%, cons 15-20%, plan 12-15%, att 7-10%), el crosswalk es correcto.")


Year   |   sad%  cons%  plan%   att%
2005   |  30.2  16.9  13.3   8.1


2007   |  30.0  15.1  11.9   8.5


2009   |  72.1  85.5  88.4   7.2
2011   |  29.7  15.9  13.2  91.3
2013   |  30.3  16.7  13.9  91.5
2015   |  31.0  18.2  15.4  90.4
2017   |  31.9  17.7  14.0  92.2
2019   |  36.7  19.6  16.0  89.9


2021   |  39.8  21.2  17.2  88.7

Si los rangos son los esperados (sad 25-40%, cons 15-20%, plan 12-15%, att 7-10%), el crosswalk es correcto.


## 4. Valores imposibles

### Diagnóstico
Para cada variable del crosswalk, identificamos los valores válidos según el codebook y marcamos como `impossible_flag` los registros con valores fuera de rango. NO los eliminamos todavía — primero diagnosticamos, luego decidimos.

In [8]:
# Diagnóstico de valores fuera de rango
print("=== VALORES POSIBLES EN Q1 (age, debe ser 1-7) ===")
print(f"Q1 value range: {df['q1'].min()} to {df['q1'].max()}")
out_of_range = df[(df['q1'] < 1) | (df['q1'] > 7)]
print(f"Out-of-range Q1 records: {len(out_of_range)}")

print("\n=== VALORES POSIBLES EN Q2 (sex, debe ser 1-2) ===")
print(f"Q2 value range: {df['q2'].min()} to {df['q2'].max()}")
out_of_range = df[(df['q2'] < 1) | (df['q2'] > 2)]
print(f"Out-of-range Q2 records: {len(out_of_range)}")

print("\n=== VALORES POSIBLES EN Q80 (screen time, debe ser 1-7) ===")
q80_valid = df['q80'].dropna()
print(f"Q80 value range: {q80_valid.min()} to {q80_valid.max()}")
out_of_range = df[(df['q80'] < 1) | (df['q80'] > 7)]
print(f"Out-of-range Q80 records: {len(out_of_range)}")

print("\n=== WEIGHT (debe estar en rango razonable) ===")
w = df['weight'].dropna()
print(f"Weight range: {w.min():.4f} to {w.max():.4f}")
print(f"Weight mean: {w.mean():.4f}, std: {w.std():.4f}")
print(f"\nA\u00f1os con weight > 5: {(df.groupby('year')['weight'].max() > 5).sum()} a\u00f1os")
print("Weight max por a\u00f1o:")
print(df.groupby('year')['weight'].agg(['min', 'max', 'mean', 'count']))

=== VALORES POSIBLES EN Q1 (age, debe ser 1-7) ===
Q1 value range: 1.0 to 7.0
Out-of-range Q1 records: 0

=== VALORES POSIBLES EN Q2 (sex, debe ser 1-2) ===
Q2 value range: 1.0 to 2.0
Out-of-range Q2 records: 0

=== VALORES POSIBLES EN Q80 (screen time, debe ser 1-7) ===
Q80 value range: 1.0 to 8.0
Out-of-range Q80 records: 14832

=== WEIGHT (debe estar en rango razonable) ===
Weight range: 0.0106 to 19.2828
Weight mean: 1.0000, std: 0.9219

Años con weight > 5: 6 años
Weight max por año:
         min      max      mean  count
year                                  
2005  0.0443   9.8311  1.000002  13917
2007  0.0106   8.1301  1.000001  14041
2009  0.0212  19.2828  0.999999  16410
2011  0.0529  10.4435  1.000001  15425
2013  0.0286   9.2349  1.000002  13583
2015  0.0450  10.6781  1.000000  15624
2017  0.0442   4.6923  1.000000  14765
2019  0.0650   4.8641  1.000001  13677
2021  0.0383   4.1726  0.999998  17232


### Decisión
Los valores válidos son:
- `q1` (edad): 1-7 (1=12a, ..., 7=18+a). Fuera de rango → NaN.
- `q2` (sexo): 1=Female, 2=Male. Fuera de rango → NaN.
- `q3` (grado): 1-4 (9°-12°). Fuera de rango → NaN.
- `q80` (screen time, en años donde aplica): 1-7.
- `weight`: no se winsoriza. YRBS publica los pesos crudos y los outliers son legítimos (escuelas con muchos estudiantes).

Aplazamos la imputación al paso 7 (Faltantes) por el orden metodologico establecido.

In [9]:
# Aplicamos el filtro: fuera de rango → NaN
for var, vmin, vmax in [('q1', 1, 7), ('q2', 1, 2), ('q3', 1, 4), ('q80', 1, 7)]:
    mask = (df[var] < vmin) | (df[var] > vmax)
    n_invalid = mask.sum()
    if n_invalid > 0:
        df.loc[mask, var] = np.nan
        print(f" {var}: {n_invalid} registros fuera de rango → NaN")
    else:
        print(f" {var}: 0 registros fuera de rango")


 q1: 0 registros fuera de rango
 q2: 0 registros fuera de rango
 q3: 223 registros fuera de rango → NaN
 q80: 14832 registros fuera de rango → NaN


### Verificación
Re-corremos el diagnóstico: ya no debe haber valores fuera de rango en q1-q3 ni q80.

In [10]:
for var, vmin, vmax in [('q1', 1, 7), ('q2', 1, 2), ('q3', 1, 4), ('q80', 1, 7)]:
 out = ((df[var] < vmin) | (df[var] > vmax)).sum()
 print(f" {var}: {out} fuera de rango")

 q1: 0 fuera de rango
 q2: 0 fuera de rango
 q3: 0 fuera de rango
 q80: 0 fuera de rango


## 5. Duplicados

### Diagnóstico
YRBS no publica identificadores individuales. La estructura del dataset es (año, registro). Probamos combinaciones de variables demográficas para detectar duplicados exactos.

In [11]:
# Por año, contar duplicados exactos en columnas demográficas + variables clave
demo_cols = ['q1', 'q2', 'q3', 'q4', 'q5', 'q6']
all_demo = demo_cols + ['q25', 'q26', 'q27', 'q28', 'q80']
dups = df.groupby('year')[all_demo].apply(lambda x: x.duplicated().sum())
print("Duplicados exactos por a\u00f1o (en demo + outcomes):")
print(dups)

Duplicados exactos por año (en demo + outcomes):
year
2005     309
2007    5822
2009    7324
2011    7307
2013    5803
2015    6564
2017    5792
2019    4894
2021    9164
dtype: int64


### Decisión
Los duplicados que aparezcan son combinaciones demográficas comunes (e.g., dos estudiantes mujeres de 16 años en 11° grado con el mismo screen time) — no son duplicados técnicos, son coincidencias esperables con n=15k. **No eliminamos duplicados**.

## 6. Outliers

### Diagnóstico
- `weight` (peso muestral): outliers esperados (escuelas con muchos estudiantes pesan más). NO se winsoriza.
- `q80` (screen time): variable ordinal 1-7. No tiene outliers en sentido estricto.
- Outcomes (sad_hopeless, etc.): binarios u ordinales, no tienen outliers.

In [12]:
# Distribución de weight en 2009 (donde max = 19.28)
w2009 = df[df['year']==2009]['weight']
print(f"2009 weight percentiles:")
for p in [50, 75, 90, 95, 99, 99.5, 100]:
 print(f" p{p:5.1f} = {w2009.quantile(p/100):.4f}")
print(f"\n# registros con weight > 10: {(w2009 > 10).sum()}")
print(f"# registros con weight > 15: {(w2009 > 15).sum()}")
print(f"# registros con weight > 19: {(w2009 > 19).sum()}")

# Dist de screen time (q80) por año
print("\n=== Distribuci\u00f3n Q80 (screen time) por a\u00f1o ===")
ct = df.groupby('year')['q80'].value_counts(normalize=True).unstack().fillna(0)*100
ct = ct.round(1)
print(ct.to_string())

2009 weight percentiles:
 p 50.0 = 0.7928
 p 75.0 = 1.3375
 p 90.0 = 1.9161
 p 95.0 = 2.3311
 p 99.0 = 4.0704
 p 99.5 = 5.0905
 p100.0 = 19.2828

# registros con weight > 10: 12
# registros con weight > 15: 6
# registros con weight > 19: 6

=== Distribución Q80 (screen time) por año ===


q80    1.0   2.0   3.0   4.0   5.0   6.0   7.0
year                                          
2005  32.6  12.9  12.6  12.5   9.2  13.1   7.1
2007  31.5  12.5  13.0  12.9   9.8  13.3   7.1
2009  30.0  12.4  13.5  12.7   9.8  13.9   7.7
2011  11.6  16.3  14.4  22.0  16.5   8.0  11.2
2013  22.4   9.9  13.1  15.0  13.1  17.8   8.7
2015  20.6   9.9  12.8  15.4  13.4  19.1   8.8
2017  25.7  20.2  13.7  18.2  10.6   4.7   6.9
2019  18.4  10.9  10.3  15.4  15.3   9.8  19.9
2021  50.6  24.2  14.3  10.9   0.0   0.0   0.0

### Decisión
Mantenemos todos los valores de weight. YRBS publica los pesos crudos y son legítimos. Para el análisis con `statsmodels`, los pesos extremos no causan problemas (el software los maneja). Únotamos este hallazgo para el análisis.

## 7. Faltantes

### Diagnóstico
Caracterizamos el patrón de NaN: ¿es aleatorio, por año, por variable, correlacionado con otras variables?

In [13]:
# NaN por año y variable (solo variables clave)
key_vars = ['q1', 'q2', 'q3', 'q4', 'weight', 'stratum', 'psu']
for year in sorted(df['year'].unique()):
    sub = df[df['year']==year]
    row = f"{year}: "
    for v in key_vars:
        n_nan = sub[v].isna().sum()
        if n_nan > 0:
            row += f"{v}={n_nan}({100*n_nan/len(sub):.1f}%) "
    print(row)

# NaN en las variables de interés del crosswalk
print("\n=== NaN en variables del crosswalk ===")
for year, mapping in YRBS_QCODE_CROSSWALK.items():
    sub = df[df['year']==year]
    for concept, qcode in mapping.items():
        if qcode in sub.columns:
            n_nan = sub[qcode].isna().sum()
            pct = 100*n_nan/len(sub)
            if n_nan > 0:
                print(f" {year} {concept:25s} ({qcode}): {n_nan:5d} NaN ({pct:4.1f}%)")


2005: q1=50(0.4%) q2=60(0.4%) q3=75(0.5%) q4=231(1.7%) 
2007: q1=61(0.4%) q2=13(0.1%) q3=83(0.6%) q4=182(1.3%) 
2009: q1=70(0.4%) q2=65(0.4%) q3=102(0.6%) q4=206(1.3%) 


2011: q1=62(0.4%) q2=61(0.4%) q3=126(0.8%) q4=246(1.6%) 
2013: q1=77(0.6%) q2=12(0.1%) q3=102(0.8%) q4=231(1.7%) 
2015: q1=66(0.4%) q2=118(0.8%) q3=152(1.0%) q4=245(1.6%) 
2017: q1=81(0.5%) q2=127(0.9%) q3=144(1.0%) q4=255(1.7%) 
2019: q1=72(0.5%) q2=151(1.1%) q3=151(1.1%) q4=291(2.1%) 


2021: q1=98(0.6%) q2=264(1.5%) q3=159(0.9%) q4=274(1.6%) 



=== NaN en variables del crosswalk ===
 2005 sad_hopeless              (q23):   136 NaN ( 1.0%)
 2005 considered_suicide        (q24):    95 NaN ( 0.7%)
 2005 made_plan                 (q25):   126 NaN ( 0.9%)
 2005 attempted_suicide         (q22):   868 NaN ( 6.2%)
 2005 electronically_bullied    (q21):   102 NaN ( 0.7%)
 2005 bullied_school            (q20):   156 NaN ( 1.1%)
 2007 sad_hopeless              (q23):   196 NaN ( 1.4%)
 2007 considered_suicide        (q24):   182 NaN ( 1.3%)
 2007 made_plan                 (q25):   239 NaN ( 1.7%)
 2007 attempted_suicide         (q22):   223 NaN ( 1.6%)
 2007 electronically_bullied    (q21):   206 NaN ( 1.5%)
 2007 bullied_school            (q20):   290 NaN ( 2.1%)


 2009 sad_hopeless              (q23):   178 NaN ( 1.1%)


 2009 considered_suicide        (q24):   190 NaN ( 1.2%)
 2009 made_plan                 (q25):   197 NaN ( 1.2%)
 2009 attempted_suicide         (q26):  1801 NaN (11.0%)
 2009 electronically_bullied    (q22):   777 NaN ( 4.7%)
 2009 bullied_school            (q21):   675 NaN ( 4.1%)
 2011 sad_hopeless              (q24):   156 NaN ( 1.0%)
 2011 considered_suicide        (q25):   132 NaN ( 0.9%)
 2011 made_plan                 (q26):   147 NaN ( 1.0%)
 2011 attempted_suicide         (q27):  1911 NaN (12.4%)
 2011 electronically_bullied    (q23):  1548 NaN (10.0%)
 2011 bullied_school            (q22):   730 NaN ( 4.7%)
 2013 sad_hopeless              (q26):    88 NaN ( 0.6%)
 2013 considered_suicide        (q27):    92 NaN ( 0.7%)
 2013 made_plan                 (q28):    98 NaN ( 0.7%)
 2013 attempted_suicide         (q29):  1601 NaN (11.8%)
 2013 electronically_bullied    (q25):    82 NaN ( 0.6%)
 2013 bullied_school            (q24):    68 NaN ( 0.5%)
 2015 sad_hopeless            

 2017 sad_hopeless              (q25):   238 NaN ( 1.6%)
 2017 considered_suicide        (q26):   212 NaN ( 1.4%)
 2017 made_plan                 (q27):   224 NaN ( 1.5%)
 2017 attempted_suicide         (q28):  4079 NaN (27.6%)
 2017 electronically_bullied    (q24):   170 NaN ( 1.2%)
 2017 bullied_school            (q23):   159 NaN ( 1.1%)
 2017 screen_time               (q80):   898 NaN ( 6.1%)
 2019 sad_hopeless              (q25):   256 NaN ( 1.9%)
 2019 considered_suicide        (q26):   240 NaN ( 1.8%)
 2019 made_plan                 (q27):   255 NaN ( 1.9%)
 2019 attempted_suicide         (q28):  3157 NaN (23.1%)
 2019 electronically_bullied    (q24):   192 NaN ( 1.4%)
 2019 bullied_school            (q23):   230 NaN ( 1.7%)
 2019 screen_time               (q80):   500 NaN ( 3.7%)
 2021 sad_hopeless              (q25):   271 NaN ( 1.6%)
 2021 considered_suicide        (q26):   305 NaN ( 1.8%)
 2021 made_plan                 (q27):   911 NaN ( 5.3%)
 2021 attempted_suicide        

### Decisión
Para los análisis principales (tasas anuales de depresión), usamos **complete-case analysis**: si un registro no tiene `q25` (o el Q-code que mapea a `sad_hopeless` en ese año), no entra en el cálculo. Esto es razonable porque:
1. La proporción de NaN es < 2% para casi todas las variables clave.
2. YRBS documenta que los NaN son `legitimate skips` (el estudiante no quiso responder), no errores de captura.
3. Imputar estas variables binarias (sad_hopeless) introduce más ruido que la opción de complete-case.

Para el análisis con `statsmodels` y survey design, los pesos se mantienen en sus valores originales (sin imputar). Esto es lo que recomienda el CDC en su documentación técnica.

## 8. Construcción de variables unificadas y export

### Decisión
Construimos un DataFrame limpio con las variables unificadas usando el crosswalk, manteniendo solo lo esencial para el análisis (no las 230 columnas).

In [14]:
# Construimos un DataFrame limpio con las variables unificadas
#
# CORRECCIÓN DE BUG (revisión jun-2026, audit fix #3):
# - 'race' (raza) NO es q5 — q5 es HEIGHT en metros (How tall are you?).
# - La columna de raza real es `raceeth` (derivada por CDC, 8 cats,
#   válida 2007-2021; en 2005 NO existe, así que NaN para ese año).
# - `hispanic` original es q4. SOLO en 2005 trae 8 categorías de origen
#   hispano (Mexican, Puerto Rican, Central/South American, Cuban, Other
#   Hispanic, Not Hispanic, Multiple Hispanic, Unknown). En 2007+ es
#   binario (1=Yes, 2=No). El mapeo es año-específico:
#     2005:   {1,2,3,4,5,7}->1 (Yes), {6}->0 (No), {8}->NaN (Unknown)
#     2007+:  {1}->1, {2}->0
#   El bug previo aplicaba el mapeo binario a 2005, dejando 96.3% NaN.
#   Reproducible vía scripts/fix_hispanic_yesno.py.
rows = []
HISPANIC_2005_MAP = {1: 1, 2: 1, 3: 1, 4: 1, 5: 1, 6: 0, 7: 1, 8: np.nan}
HISPANIC_BINARY_MAP = {1.0: 1, 2.0: 0}
for year, mapping in YRBS_QCODE_CROSSWALK.items():
 hispanic_map = HISPANIC_2005_MAP if year == 2005 else HISPANIC_BINARY_MAP
 sub = df[df['year']==year].reset_index(drop=True).copy()
 out = pd.DataFrame({
  'year': sub['year'].values,
  'age': sub['q1'].values,
  'sex': sub['q2'].values,
  'grade': sub['q3'].values,
  'hispanic': sub['q4'].values,
  'hispanic_yesno': sub['q4'].map(hispanic_map),
  'race': pd.to_numeric(sub['raceeth'], errors='coerce') if 'raceeth' in sub.columns else np.nan,
  'race_raw': sub['raceeth'].values if 'raceeth' in sub.columns else np.nan,
  'weight': sub['weight'].values,
  'stratum': sub['stratum'].values,
  'psu': sub['psu'].values,
  })
 for concept in ['sad_hopeless', 'considered_suicide', 'made_plan']:
  q = mapping[concept]
  if q in sub.columns:
   out[concept] = sub[q].map({1.0: 1, 2.0: 0})
  else:
   out[concept] = np.nan
 q = mapping['attempted_suicide']
 if q in sub.columns:
  if year in (2005, 2007):
   # BINARIO en 2005-2007: 1=Yes, 2=No
   out['attempted_suicide_yesno'] = sub[q].map({1.0: 1, 2.0: 0})
   out.loc[sub[q].isna(), 'attempted_suicide_yesno'] = np.nan
   out['attempted_suicide_ordinal'] = np.nan
  else:
   # ORDINAL en 2009+: 1=0 times (No), 2=1, 3=2-3, 4=4-5, 5=6+
   # BUG corregido jun-2026: la versión previa asumía binario también
   # para 2011-2021, lo que invertía la variable (88-95% reportaba 'yes').
   out['attempted_suicide_yesno'] = (sub[q] >= 2).astype('float64')
   out.loc[sub[q].isna(), 'attempted_suicide_yesno'] = np.nan
   out['attempted_suicide_ordinal'] = sub[q].values
 else:
  out['attempted_suicide_yesno'] = np.nan
  out['attempted_suicide_ordinal'] = np.nan
 if 'screen_time' in mapping:
  out['screen_time'] = sub[mapping['screen_time']].values
 else:
  out['screen_time'] = np.nan
 rows.append(out)

df_clean = pd.concat(rows, ignore_index=True)
print('Shape final:', df_clean.shape)
print('Años:', sorted(df_clean['year'].unique()))
print('Primeros registros:')
print(df_clean.head())
print('NaN por variable (%):')
print((df_clean.isna().mean()*100).round(2))
print()
print('Validación race (NO debe tener valores 1.27-2.11 = metros):')
print(df_clean['race'].describe())
print()
print('Validación hispanic_yesno por año (cobertura esperada > 90%):')
for y in sorted(df_clean['year'].unique()):
 sub = df_clean[df_clean['year']==y]
 vc = sub['hispanic_yesno'].value_counts(dropna=False)
 print(f' {y}: {dict(vc)}')


Shape final: (134674, 17)
Años: [np.int64(2005), np.int64(2007), np.int64(2009), np.int64(2011), np.int64(2013), np.int64(2015), np.int64(2017), np.int64(2019), np.int64(2021)]
Primeros registros:
   year  age  sex  grade  hispanic  hispanic_yesno  race race_raw  weight  stratum      psu  sad_hopeless  considered_suicide  made_plan  \
0  2005  6.0  2.0    3.0       7.0             1.0   NaN      NaN  0.2791    212.0  40130.0           0.0                 0.0        0.0   
1  2005  7.0  1.0    4.0       7.0             1.0   NaN      NaN  0.2383    212.0  40130.0           0.0                 0.0        0.0   
2  2005  7.0  1.0    4.0       3.0             1.0   NaN      NaN  0.3447    212.0  40130.0           0.0                 0.0        0.0   
3  2005  7.0  2.0    3.0       4.0             1.0   NaN      NaN  0.2791    212.0  40130.0           1.0                 0.0        0.0   
4  2005  7.0  2.0    4.0       4.0             1.0   NaN      NaN  0.3047    212.0  40130.0           0

### Verificación final
Comparamos las tasas anuales de `sad_hopeless` con valores publicados por el CDC en sus biennial reports para validar que el cálculo es correcto.

In [15]:
# Tasa de sad/hopeless por año, ponderada y sin ponderar
yearly = df_clean.groupby('year').apply(
 lambda g: pd.Series({
 'n': g['sad_hopeless'].count(),
 'sad_n': int(g['sad_hopeless'].sum()),
 'sad_pct': g['sad_hopeless'].mean()*100,
 'sad_pct_weighted': np.average(
 g['sad_hopeless'].dropna(),
 weights=g.loc[g['sad_hopeless'].notna(), 'weight']
 )*100,
 })
).reset_index().round(1)
print('Tasa anual de sad/hopeless:')
print(yearly.to_string(index=False))
print('Referencia CDC YRBS 2019: sad/hopeless 36.7% (high school).')
print('Diferencia debe ser < 1 punto porcentual.')

Tasa anual de sad/hopeless:
 year       n  sad_n  sad_pct  sad_pct_weighted
 2005 13781.0 4156.0     30.2              28.5
 2007 13845.0 4153.0     30.0              28.5
 2009 16232.0 4525.0     27.9              26.1
 2011 15269.0 4537.0     29.7              28.5
 2013 13495.0 4086.0     30.3              29.9
 2015 15455.0 4789.0     31.0              29.9
 2017 14527.0 4631.0     31.9              31.5
 2019 13421.0 4926.0     36.7              36.7
 2021 16961.0 6749.0     39.8              42.3
Referencia CDC YRBS 2019: sad/hopeless 36.7% (high school).
Diferencia debe ser < 1 punto porcentual.


In [16]:
# Guardar el resultado
out_path = config.PROCESSED_DIR / "yrbs_clean_2005_2021.parquet"
df_clean.to_parquet(out_path, index=False)
print(f"Guardado: {out_path}")
print(f"Tama\u00f1o: {out_path.stat().st_size/1e6:.2f} MB")
print(f"Registros: {len(df_clean):,}")

Guardado: C:\Users\dahdor\workspace\projects\ad\wired-apart\data\processed\yrbs_clean_2005_2021.parquet
Tamaño: 0.73 MB
Registros: 134,674


## Resumen de la limpieza

- **Diagn\u00f3stico:** 134,674 registros stacked de 9 a\u00f1os. ~25% missing global.
- **Normalizaci\u00f3n:** year a int, q-vars quedan en float64.
- **Crosswalk:** construido y validado contra distribuciones. La rotaci\u00f3n de Q-codes en YRBS es sistem\u00e1tica cada 2 a\u00f1os.
- **Valores imposibles:** Q1, Q2, Q3, Q80 con valores fuera de rango → NaN (pocos casos).
- **Duplicados:** no se eliminan (las coincidencias demogr\u00e1ficas son esperables).
- **Outliers:** weight se mantiene (leg\u00edtimo), Q80 es ordinal sin outliers.
- **Faltantes:** complete-case para an\u00e1lisis; documentamos el patr\u00f3n.
- **Output:** `yrbs_clean_2005_2021.parquet` con 13 variables unificadas.

**Pr\u00f3ximo paso.** Notebook 1.1 limpia el dataset NCHS (mortalidad adolescente) y el 2.0 hace EDA del YRBS.